# Converting labels from the Planet PNG to the Sentinel-2 PNG

We have the labels for the Planet images, and we need to convert them to geographic coordinates, and then to the dimensions of the Sentinel-2 images. 

In [1]:
import json
import os
import pandas as pd
import numpy as np
import rasterio

from PIL import Image
from pyproj import Transformer   
from datetime import datetime    

## Import the Planet raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [15]:
tif_base_dir = 'images/planet_2_sentinel'
tif_name = '20240604_mimal_test_planet'
tif_path = os.path.join(tif_base_dir, tif_name + '.tif')

with rasterio.open(tif_path) as raster:
    # Read the raster band
    planet_raster = raster.read(1)
    # Get the metadata of the raster
    planet_raster_meta = raster.meta
    # Get the raster transform parameters
    planet_raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(planet_raster.shape)
print("\n")

print("Raster metadata:")
print(planet_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(planet_raster_transform)

Shape of the raster (rows, columns):
(11309, 14180)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint16', 'nodata': 0.0, 'width': 14180, 'height': 11309, 'count': 4, 'crs': CRS.from_epsg(32753), 'transform': Affine(3.0, 0.0, 438081.0,
       0.0, -3.0, 8533512.0)}


Affine transformation parameters:
| 3.00, 0.00, 438081.00|
| 0.00,-3.00, 8533512.00|
| 0.00, 0.00, 1.00|


## Account for the png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [3]:
Image.MAX_IMAGE_PIXELS = 933120000 # change to greater than the nnumber of pixels (only needed if there's a warning)

# Base directory for the png image
png_base_dir = 'images/planet_2_sentinel'

# Load the png image
image = Image.open(f'{png_base_dir}/20240604_mimal_test_planet.png')

# Convert the image to a numpy array
pixel_data = np.array(image)

### Get Planet image information

In [16]:
# Get image information
png_width, png_height = image.size
print(f'Image size: ({png_width}, {png_height})')

# Calculate the different in the width and height of the image and the raster
width_diff = png_width - planet_raster.shape[1]
height_diff = png_height - planet_raster.shape[0]
print(f'Difference in width: {width_diff}')
print(f'Difference in height: {height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
width_diff = width_diff - min_pad
height_diff = height_diff - min_pad
print(f'Difference in width after removing right padding: {width_diff}')
print(f'Difference in height after removing bottom padding: {height_diff}')

Image size: (14976, 12064)
Difference in width: 796
Difference in height: 755
Difference in width after removing right padding: 484
Difference in height after removing bottom padding: 443


# Import the labels for the Planet PNG

In [10]:
# Path to the LabelMe JSON file
labelme_json_path = os.path.join(png_base_dir, '20240604_mimal_test_planet.json')

# Load the JSON file
with open(labelme_json_path, 'r') as f:
    labelme_data = json.load(f)

# Extract bounding box labels
bounding_boxes = []
for shape in labelme_data['shapes']:
    label = shape['label']
    points = shape['points']
    x_min, y_min = points[0]
    x_max, y_max = points[1]
    bounding_boxes.append({'label': label, 'x_min': x_min, 'y_min': y_min, 'x_max': x_max, 'y_max': y_max})

# Print the bounding boxes
print(bounding_boxes)

[{'label': 'WH_wet', 'x_min': 2973.1895223420647, 'y_min': 10518.181818181818, 'x_max': 3030.0462249614793, 'y_max': 10563.328197226501}, {'label': 'Dry_WH', 'x_min': 2431.790322580645, 'y_min': 10974.935483870968, 'x_max': 2495.967741935484, 'y_max': 11007.258064516129}, {'label': 'WH_swamp', 'x_min': 3335.075091575092, 'y_min': 9852.563186813186, 'x_max': 3575.1428571428573, 'y_max': 9965.714285714284}, {'label': 'WH_swamp', 'x_min': 5316.9473684210525, 'y_min': 4770.742690058479, 'x_max': 5758.000000000001, 'y_max': 5048.181818181818}, {'label': 'WH_swamp', 'x_min': 5328.327102803739, 'y_min': 4290.03313508921, 'x_max': 5655.272727272728, 'y_max': 4509.090909090909}, {'label': 'WH_wet', 'x_min': 6071.770491803279, 'y_min': 2261.311475409836, 'x_max': 6285.213114754098, 'y_max': 2376.7213114754095}, {'label': 'WH_swamp', 'x_min': 2533.2915533637934, 'y_min': 2910.21764388349, 'x_max': 2711.3300492610833, 'y_max': 2997.5172413793102}, {'label': 'WH_swamp', 'x_min': 4572.290540540541, 

In [ ]:
def convert_pixel_to_geo_coords(bboxes, raster_transform, width_diff, height_diff):
    """
    Convert pixel coordinates from bounding boxes to geographic coordinates.
    
    Args:
        bboxes (list): List of dictionaries containing bounding box coordinates
        raster_transform (affine.Affine): Affine transformation object
        width_diff (int): Padding on the left side of the image
        height_diff (int): Padding on the top side of the image
        
    Returns:
        list: List of dictionaries with bounding boxes in geographic coordinates
    """
    geo_boxes = []
    
    for bbox in bboxes:
        
        # Remove the padding from the pixel coordinates
        x_min_px = bbox['x_min'] - width_diff
        y_min_px = bbox['y_min'] - height_diff
        x_max_px = bbox['x_max'] - width_diff
        y_max_px = bbox['y_max'] - height_diff
        
        # Convert pixel coordinates to geographic coordinates using the raster transform
        # The * operator applies the transform to the (x, y) coordinates
        x_min_geo, y_min_geo = raster_transform * (x_min_px, y_min_px)
        x_max_geo, y_max_geo = raster_transform * (x_max_px, y_max_px)
        
        # Create a new dictionary with the geographic coordinates
        geo_box = {
            'label': bbox['label'],
            'x_min_geo': x_min_geo,
            'y_min_geo': y_min_geo,
            'x_max_geo': x_max_geo,
            'y_max_geo': y_max_geo,
            # Keep the original pixel coordinates for reference
            'x_min_px': bbox['x_min'],
            'y_min_px': bbox['y_min'],
            'x_max_px': bbox['x_max'],
            'y_max_px': bbox['y_max']
        }
        
        geo_boxes.append(geo_box)
    
    return geo_boxes

# Convert all bounding boxes to geographic coordinates
geo_coordinates = convert_pixel_to_geo_coords(bounding_boxes, planet_raster_transform, width_diff, height_diff)

# Display the first 5 converted coordinates
for i, box in enumerate(geo_coordinates[:5]):
    print(f"Box {i+1} - Label: {box['label']}")
    print(f"  Pixel: ({box['x_min_px']}, {box['y_min_px']}) to ({box['x_max_px']}, {box['y_max_px']})")
    print(f"  Geo: ({box['x_min_geo']}, {box['y_min_geo']}) to ({box['x_max_geo']}, {box['y_max_geo']})")
    print("")

Box 1 - Label: WH_wet
  Pixel: (2973.1895223420647, 10518.181818181818) to (3030.0462249614793, 10563.328197226501)
  Geo: (445548.5685670262, 8503286.454545455) to (445719.13867488445, 8503151.01540832)

Box 2 - Label: Dry_WH
  Pixel: (2431.790322580645, 10974.935483870968) to (2495.967741935484, 11007.258064516129)
  Geo: (443924.37096774194, 8501916.193548387) to (444116.9032258064, 8501819.225806452)

Box 3 - Label: WH_swamp
  Pixel: (3335.075091575092, 9852.563186813186) to (3575.1428571428573, 9965.714285714284)
  Geo: (446634.2252747253, 8505283.31043956) to (447354.4285714286, 8504943.857142856)

Box 4 - Label: WH_swamp
  Pixel: (5316.9473684210525, 4770.742690058479) to (5758.000000000001, 5048.181818181818)
  Geo: (452579.84210526315, 8520528.771929825) to (453903.0, 8519696.454545455)

Box 5 - Label: WH_swamp
  Pixel: (5328.327102803739, 4290.03313508921) to (5655.272727272728, 4509.090909090909)
  Geo: (452613.9813084112, 8521970.900594732) to (453594.8181818182, 8521313.72

## Import the Sentinel-2 raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [18]:
tif_base_dir = 'images/planet_2_sentinel/tifs'
tif_name = 'mimal_test_2024-06'
tif_path = os.path.join(tif_base_dir, tif_name + '.tif')

with rasterio.open(tif_path) as raster:
    # Read the raster band
    S2_raster = raster.read(1)
    # Get the metadata of the raster
    S2_raster_meta = raster.meta
    # Get the raster transform parameters
    S2_raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(S2_raster.shape)
print("\n")

print("Raster metadata:")
print(S2_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(S2_raster_transform)

Shape of the raster (rows, columns):
(3422, 4380)


Raster metadata:
{'driver': 'GTiff', 'dtype': 'uint8', 'nodata': None, 'width': 4380, 'height': 3422, 'count': 3, 'crs': CRS.from_epsg(4326), 'transform': Affine(8.983152841195215e-05, 0.0, 134.42767203983848,
       0.0, -8.983152841195215e-05, -13.26479297989409)}


Affine transformation parameters:
| 0.00, 0.00, 134.43|
| 0.00,-0.00,-13.26|
| 0.00, 0.00, 1.00|


## Convert the labels in the json file to the same projection as the Planet tif file

As the labels are in pixel coordinates, we need to:
- convert them to the same projection as the Planet tif file - CRS.from_epsg(32753)
- convert them to the projection of the Sentinel-2 data - CRS.from_epsg(4326)
- convert them to the Sentinel-2 pixel coordinates using the raster transformation from the S2 metadata.

In [19]:
def convert_to_sentinel_coords(geo_boxes, S2_raster_transform):
    """
    Convert bounding boxes from UTM coordinates (EPSG:32753) to WGS84 (EPSG:4326)
    and then to Sentinel-2 pixel coordinates.
    
    Args:
        geo_boxes (list): List of dictionaries with bounding box coordinates in UTM (EPSG:32753)
        S2_raster_transform (affine.Affine): Affine transformation of the Sentinel-2 raster
        
    Returns:
        list: List of dictionaries with bounding boxes in Sentinel-2 pixel coordinates
    """
    # Create transformer from Planet CRS (EPSG:32753) to Sentinel-2 CRS (EPSG:4326)
    utm_to_wgs84 = Transformer.from_crs('epsg:32753', 'epsg:4326', always_xy=True)
    
    s2_boxes = []
    
    for box in geo_boxes:
        # Convert UTM coordinates to WGS84
        x_min_wgs84, y_min_wgs84 = utm_to_wgs84.transform(box['x_min_geo'], box['y_min_geo'])
        x_max_wgs84, y_max_wgs84 = utm_to_wgs84.transform(box['x_max_geo'], box['y_max_geo'])
        
        # Convert WGS84 to Sentinel-2 pixel coordinates using the inverse transform
        # The ~ operator inverts the affine transform
        x_min_s2_px, y_min_s2_px = ~S2_raster_transform * (x_min_wgs84, y_min_wgs84)
        x_max_s2_px, y_max_s2_px = ~S2_raster_transform * (x_max_wgs84, y_max_wgs84)
        
        # Create a new dictionary with the Sentinel-2 pixel coordinates
        s2_box = {
            'label': box['label'],

            # Store WGS84 coordinates
            'x_min_wgs84': x_min_wgs84,
            'y_min_wgs84': y_min_wgs84,
            'x_max_wgs84': x_max_wgs84,
            'y_max_wgs84': y_max_wgs84,

            # Store Sentinel-2 pixel coordinates
            'x_min_s2_px': x_min_s2_px,
            'y_min_s2_px': y_min_s2_px,
            'x_max_s2_px': x_max_s2_px,
            'y_max_s2_px': y_max_s2_px,
            
            # Keep original UTM and Planet pixel coordinates
            'x_min_utm': box['x_min_geo'],
            'y_min_utm': box['y_min_geo'],
            'x_max_utm': box['x_max_geo'],
            'y_max_utm': box['y_max_geo'],
            'x_min_planet_px': box['x_min_px'],
            'y_min_planet_px': box['y_min_px'],
            'x_max_planet_px': box['x_max_px'],
            'y_max_planet_px': box['y_max_px']
        }
        
        s2_boxes.append(s2_box)
    
    return s2_boxes

# Convert the bounding boxes to Sentinel-2 coordinates
sentinel2_boxes = convert_to_sentinel_coords(geo_coordinates, S2_raster_transform)

# Display the first 5 converted coordinates
for i, box in enumerate(sentinel2_boxes[:5]):
    print(f"Box {i+1} - Label: {box['label']}")
    print(f"  Planet Pixel: ({box['x_min_planet_px']:.1f}, {box['y_min_planet_px']:.1f}) to ({box['x_max_planet_px']:.1f}, {box['y_max_planet_px']:.1f})")
    print(f"  UTM (EPSG:32753): ({box['x_min_utm']:.1f}, {box['y_min_utm']:.1f}) to ({box['x_max_utm']:.1f}, {box['y_max_utm']:.1f})")
    print(f"  WGS84 (EPSG:4326): ({box['x_min_wgs84']:.6f}, {box['y_min_wgs84']:.6f}) to ({box['x_max_wgs84']:.6f}, {box['y_max_wgs84']:.6f})")
    print(f"  Sentinel-2 Pixel: ({box['x_min_s2_px']:.1f}, {box['y_min_s2_px']:.1f}) to ({box['x_max_s2_px']:.1f}, {box['y_max_s2_px']:.1f})")
    print("")

Box 1 - Label: WH_wet
  Planet Pixel: (2973.2, 10518.2) to (3030.0, 10563.3)
  UTM (EPSG:32753): (445548.6, 8503286.5) to (445719.1, 8503151.0)
  WGS84 (EPSG:4326): (134.496771, -13.538228) to (134.498345, -13.539456)
  Sentinel-2 Pixel: (769.2, 3043.9) to (786.7, 3057.5)

Box 2 - Label: Dry_WH
  Planet Pixel: (2431.8, 10974.9) to (2496.0, 11007.3)
  UTM (EPSG:32753): (443924.4, 8501916.2) to (444116.9, 8501819.2)
  WGS84 (EPSG:4326): (134.481735, -13.550587) to (134.483512, -13.551468)
  Sentinel-2 Pixel: (601.8, 3181.4) to (621.6, 3191.2)

Box 3 - Label: WH_swamp
  Planet Pixel: (3335.1, 9852.6) to (3575.1, 9965.7)
  UTM (EPSG:32753): (446634.2, 8505283.3) to (447354.4, 8504943.9)
  WGS84 (EPSG:4326): (134.506842, -13.520192) to (134.513491, -13.523275)
  Sentinel-2 Pixel: (881.3, 2843.1) to (955.3, 2877.4)

Box 4 - Label: WH_swamp
  Planet Pixel: (5316.9, 4770.7) to (5758.0, 5048.2)
  UTM (EPSG:32753): (452579.8, 8520528.8) to (453903.0, 8519696.5)
  WGS84 (EPSG:4326): (134.562035, 

## Account for the Sentinel-2 png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [20]:
Image.MAX_IMAGE_PIXELS = 933120000 # change to greater than the nnumber of pixels (only needed if there's a warning)

# Base directory for the png image
png_base_dir = 'images/planet_2_sentinel'

# Load the png image
S2_image = Image.open(f'{png_base_dir}/mimal_test_S2_2024-06.png')

# Convert the image to a numpy array
pixel_data = np.array(S2_image)

### Get Sentinel-2 image information

In [24]:
# Get image information
S2_png_width, S2_png_height = S2_image.size
print(f'Image size: ({S2_png_width}, {S2_png_height})')

# Calculate the different in the width and height of the image and the raster
S2_width_diff = S2_png_width - S2_raster.shape[1]
S2_height_diff = S2_png_height - S2_raster.shape[0]
print(f'Difference in width: {S2_width_diff}')
print(f'Difference in height: {S2_height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
S2_width_diff = S2_width_diff - min_pad
S2_height_diff = S2_height_diff - min_pad
print(f'Difference in S2 width after removing right padding: {S2_width_diff}')
print(f'Difference in S2 height after removing bottom padding: {S2_height_diff}')

Image size: (5408, 4160)
Difference in width: 1028
Difference in height: 738
Difference in S2 width after removing right padding: 716
Difference in S2 height after removing bottom padding: 426


In [26]:
def save_labelme_json(json_data, filepath):
    """
    Save LabelMe JSON data to a file.
    
    Args:
        json_data (dict): LabelMe JSON data
        filepath (str): Path to save the JSON file
    """
    with open(filepath, 'w') as f:
        json.dump(json_data, f, indent=2)

# Function to convert the Sentinel-2 bounding boxes to LabelMe format
def convert_sentinel_boxes_to_labelme(boxes, image_path, image_height, image_width, width_diff, height_diff):
    """
    Convert Sentinel-2 bounding boxes to LabelMe JSON format.
    
    Args:
        boxes (list): List of dictionaries containing bounding box information
        image_path (str): Path to the corresponding image file
        image_height (int): Height of the image in pixels
        image_width (int): Width of the image in pixels
        width_diff (int): Width padding to add to pixel coordinates
        height_diff (int): Height padding to add to pixel coordinates
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path),
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Process each bounding box
    for box in boxes:
        # Add the padding to the pixel coordinates
        x_min = box['x_min_s2_px'] + width_diff
        y_min = box['y_min_s2_px'] + height_diff
        x_max = box['x_max_s2_px'] + width_diff
        y_max = box['y_max_s2_px'] + height_diff
        
        # Create a shape for the bounding box
        shape = {
            "label": box['label'],
            "points": [
                [float(x_min), float(y_min)],  # Top-left
                [float(x_max), float(y_max)]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

# Convert and save the Sentinel-2 bounding boxes
S2_image_path = f'{png_base_dir}/mimal_test_S2_2024-06.png'

# Convert the bounding boxes to LabelMe format
sentinel2_labelme = convert_sentinel_boxes_to_labelme(
    boxes=sentinel2_boxes,
    image_path=S2_image_path,
    image_height=S2_png_height,
    image_width=S2_png_width,
    width_diff=S2_width_diff,
    height_diff=S2_height_diff
)

# Save the LabelMe JSON file
S2_json_path = f'{png_base_dir}/mimal_test_S2_2024-06.json'
save_labelme_json(sentinel2_labelme, S2_json_path)

print(f"Sentinel-2 bounding boxes converted to LabelMe format and saved to {S2_json_path}")
print(f"Number of bounding boxes: {len(sentinel2_boxes)}")

Sentinel-2 bounding boxes converted to LabelMe format and saved to images/planet_2_sentinel/mimal_test_S2_2024-06.json
Number of bounding boxes: 187


In [6]:
# Create the reprojection function
coord_transformer = Transformer.from_crs('epsg:4326', 'epsg:32753', always_xy=True)

# Test on sample set of coordinates
x,y = coord_transformer.transform(133.6153775, -13.7976017)
print(x,y)

# Check the pixel coordinates of the transformed coordinates
pixel_column, pixel_row = ~raster_transform * (x, y)
print(pixel_column, pixel_row)

350330.60260668746 8474226.385882989
-29250.132464437513 19761.871372337453


## Read in the csv file of labelled coordinates

In [7]:
# Specify the path to your CSV file
csv_base_dir = f'data/'
# csv_name = '2024 Rapid Waterhole Assessment_aligned.csv' # with health states
csv_name = '2024 Rapid Waterhole Assessment_aligned.csv'
csv_file_path = os.path.join(csv_base_dir, csv_name)

# Read the CSV file into a DataFrame
waterhole_labelled_df = pd.read_csv(csv_file_path)

print(f'Number of samples: {len(waterhole_labelled_df)}')
print('\n')

# Display the first few rows of the DataFrame
print(waterhole_labelled_df.head())

Number of samples: 122


   OBJECTID   Timestamp   Latitude   Longitude  Class        Observer  \
0         1  2024-05-07 -13.657141  134.354081      2  Andrew Hoskins   
1         2  2024-05-07 -13.655816  134.371830      2  Andrew Hoskins   
2         3  2024-05-07 -13.655044  134.393416      3  Andrew Hoskins   
3         4  2024-05-07 -13.659928  134.417495      2  Andrew Hoskins   
4         5  2024-05-07 -13.657286  134.525346      2  Andrew Hoskins   

   ObserverSi Water_Type_Chopper Water_Type_Satellite transectNu permanence  \
0  Back Right                NaN                River        m08          P   
1  Back Right                NaN      River/Billabong        m08          P   
2  Back Right                NaN            Billabong        m08     I or E   
3  Back Right                NaN               Stream        m08     I or E   
4  Back Right                NaN            Billabong        m08     I or E   

     ID Wet_Dry_Chopper Wet_Dry_Satellite  
0   NaN          

## Reproject the label coordinates to the same projection as the tif file

In [8]:
x_proj, y_proj = coord_transformer.transform(
    waterhole_labelled_df['Longitude'].values, 
    waterhole_labelled_df['Latitude'].values
)

print(x_proj, y_proj)

[430143.17685067 432062.43965287 434396.87455498 437002.42767336
 448666.15966412 458053.23977042 458237.10489792 458987.86848127
 461874.23246752 449507.09670538 445638.36590115 444025.54981718
 440794.70240962 435105.66625879 428903.23024269 425354.55770888
 424196.77749568 423212.16039536 421165.12538497 417110.28161243
 401196.41106128 391625.28131144 373867.42304861 371360.14238773
 358132.66349237 425752.81788484 426812.24890471 429614.87363284
 458855.26207083 449006.87935519 421700.49235191 421251.19090197
 420799.34395841 389044.36386998 406364.40849872 427547.09390463
 436041.47970019 445150.68701575 459923.74064893 514935.59402967
 382948.88660257 381597.26513111 386069.49127455 388823.87049276
 390486.87336337 379662.95731606 446908.44526433 451882.41080017
 453221.3678156  453138.96858241 455277.34374629 463802.47700669
 444476.01486059 369842.88634585 370015.66197925 371192.01958601
 394621.02910232 397881.89751955 367844.88179619 366973.09283339
 366804.6126904  366446.8

## Convert the label coordinates to pixel coordinates

These pixel coordinates are in reference to the tif file, not the png yet (if it has padding).

In [9]:
# Convert to pixel coordinates
pixel_coords = np.array([~raster_transform * (x, y) for x, y in zip(x_proj, y_proj)])
print(pixel_coords)

[[ -2645.94104978  14471.10079774]
 [ -2006.18678238  14420.5911999 ]
 [ -1228.04181501  14390.12407193]
 [  -359.52410888  14568.05094087]
 [  3528.38655471  14462.20206133]
 [  6657.41325681  14518.42543212]
 [  6718.70163264  14493.14788764]
 [  6968.95616042  14613.4865288 ]
 [  7931.07748917  14430.70499074]
 [  3808.69890179  10670.483244  ]
 [  2519.12196705  10092.17430934]
 [  1981.51660573  10642.84589428]
 [   904.56746987  10696.52231285]
 [  -991.77791374  10655.48777388]
 [ -3059.25658577  10707.92841225]
 [ -4242.14743037  10726.13536447]
 [ -4628.07416811  10725.38972459]
 [ -4956.27986821  10729.28584537]
 [ -5638.62487168  10474.58410868]
 [ -6990.23946252  10818.37665637]
 [-12294.86297957  12069.48225676]
 [-15485.23956285  11520.8599326 ]
 [-21404.52565046   9914.85006453]
 [-22240.28587076   9420.73343562]
 [-26649.44550254  12061.96713864]
 [ -4109.39403839  14318.03224081]
 [ -3756.2503651   14255.47342981]
 [ -2822.04212239  14418.08043699]
 [  6924.75402361  1

## Add the new columns to the dataframe

Here is where we add to the x and y coordinates to account for the padding in the png file.

In [10]:
# Add the new columns to the dataframe
waterhole_labelled_df['x_proj'] = x_proj
waterhole_labelled_df['y_proj'] = y_proj
waterhole_labelled_df['pixel_col'] = pixel_coords[:, 0] + width_diff
waterhole_labelled_df['pixel_row'] = pixel_coords[:, 1] + height_diff

print(waterhole_labelled_df.head())

   OBJECTID   Timestamp   Latitude   Longitude  Class        Observer  \
0         1  2024-05-07 -13.657141  134.354081      2  Andrew Hoskins   
1         2  2024-05-07 -13.655816  134.371830      2  Andrew Hoskins   
2         3  2024-05-07 -13.655044  134.393416      3  Andrew Hoskins   
3         4  2024-05-07 -13.659928  134.417495      2  Andrew Hoskins   
4         5  2024-05-07 -13.657286  134.525346      2  Andrew Hoskins   

   ObserverSi Water_Type_Chopper Water_Type_Satellite transectNu permanence  \
0  Back Right                NaN                River        m08          P   
1  Back Right                NaN      River/Billabong        m08          P   
2  Back Right                NaN            Billabong        m08     I or E   
3  Back Right                NaN               Stream        m08     I or E   
4  Back Right                NaN            Billabong        m08     I or E   

     ID Wet_Dry_Chopper Wet_Dry_Satellite         x_proj        y_proj  \
0   NaN     

## Define the function to create the json file

Essentially we are just taking the pixel coordinates and the labels and creating a json file with the correct format.

In [11]:
def csv_to_labelme(x, y, 
                   labels=None, 
                   image_path=None, 
                   image_height=None, 
                   image_width=None):
    """
    Convert coordinate columns from a DataFrame to LabelMe JSON format.
    
    Args:
        x (pd.Series): Series containing x/longitude coordinates
        y (pd.Series): Series containing y/latitude coordinates
        labels (pd.Series, optional): Series containing point labels. Defaults to None
        image_path (str, optional): Path to the corresponding image file. Defaults to None
        image_height (int, optional): Height of the image in pixels. Defaults to None
        image_width (int, optional): Width of the image in pixels. Defaults to None
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Validate inputs
    if len(x) != len(y):
        raise ValueError("x and y coordinates must have the same length")
    if labels is not None and len(labels) != len(x):
        raise ValueError("labels must have the same length as coordinates")
    
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path) if image_path else "",
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Convert each point to LabelMe shape
    point_size = 5  # Size of the point representation in pixels
    
    for i in range(len(x)):
        # Skip if coordinates are NaN
        if pd.isna(x[i]) or pd.isna(y[i]):
            continue
            
        shape = {
            "label": str(labels.iloc[i]) if labels is not None else "point",
            "points": [
                [float(x[i]) - point_size, float(y[i]) - point_size],  # Top-left
                [float(x[i]) + point_size, float(y[i]) + point_size]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

## Function to save the json file

In [12]:
# Convert the DataFrame to LabelMe JSON
def save_labelme_json(labelme_json, output_path):
    """Save the LabelMe JSON to file."""
    with open(output_path, 'w') as f:
        json.dump(labelme_json, f, indent=2)

## Run the csv to json function

The image path should be the name of the png file, and the output path of the save_labelme_json function should lead to where the png is saved.

In [14]:
# Convert and save
labelme_json = csv_to_labelme(
    x=waterhole_labelled_df['pixel_col'],
    y=waterhole_labelled_df['pixel_row'],
    labels=waterhole_labelled_df['Class'],
    image_path=tif_name + '.png',
    image_height=png_height,
    image_width=png_width
)
    
# Save the LabelMe JSON file
save_labelme_json(labelme_json, f'{png_base_dir}/{tif_name}.json')
print(labelme_json)

{'version': '5.0.1', 'flags': {}, 'shapes': [{'label': '2', 'points': [[-2166.9410497775243, 14909.100797740743], [-2156.9410497775243, 14919.100797740743]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '2', 'points': [[-1527.1867823759094, 14858.591199895367], [-1517.1867823759094, 14868.591199895367]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '3', 'points': [[-749.0418150051846, 14828.124071931466], [-739.0418150051846, 14838.124071931466]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '2', 'points': [[119.47589112032438, 15006.050940865185], [129.47589112032438, 15016.050940865185]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '2', 'points': [[4007.3865547062014, 14900.202061329037], [4017.3865547062014, 14910.202061329037]], 'group_id': None, 'shape_type': 'rectangle', 'flags': {}}, {'label': '3', 'points': [[7136.4132568080095, 14956.425432116725], [7146.4132568080095, 14966.425432